### Settings

In [4]:
#imports
from pathlib import Path
import sys
sys.path.append('/Users/micahlim/Desktop/NMF_separation/NMFtoolbox/python')
import numpy as np
import librosa
from NMFtoolbox.NMFD import NMFD
from NMFtoolbox.NMF import NMF
from NMFtoolbox.utils import EPS
from mir_eval.separation import bss_eval_sources
import os
import matplotlib.pyplot as plt
import IPython.display as ipd
import soundfile as sf #library to read and write sound files
import statistics

In [24]:
#restart / delete all previous results

import shutil

for folder in ["NMFresults", "NMFDresults", "original_data_plot", "results_plot"]:
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder, exist_ok=True)

#create result folders
folders_nmf = [
    'NMFresults/static',
    'NMFresults/dynamic',
    'NMFresults/static_kick',
    'NMFresults/static_snare',
    'NMFresults/dynamic_kick',
    'NMFresults/dynamic_snare'
]

for folder in folders_nmf:
    os.makedirs(folder, exist_ok=True)


# NMFDresults folders
folders_nmfd = [
    'NMFDresults/static',
    'NMFDresults/dynamic',
    'NMFDresults/static_kick',
    'NMFDresults/static_snare',
    'NMFDresults/dynamic_kick',
    'NMFDresults/dynamic_snare'
]

for folder in folders_nmfd:
    os.makedirs(folder, exist_ok=True)

In [36]:
#parameters
sample_rate = 22050
FRAME = 512
HOP = 256
template_frames = 32
num_files = 1

### Run NMF and NMFD

In [34]:
#run NMF and NMFD on static mixtures
for static_mix_file in sorted(Path("dataset/static").glob("*.wav"), key=lambda x: int(x.stem.split('_')[-1])):
    print(static_mix_file)
    static_mix, sr = librosa.load(static_mix_file, sr=sample_rate)
    static_mix_stft = librosa.stft(static_mix, n_fft=FRAME, hop_length=HOP)
    V_static = np.abs(static_mix_stft) + EPS
    V_static_scale = V_static.sum()

    # Build parameters matching this file's spectrogram shape
    NMFDparam = {
        'numComp': 2,
        'numIter': 80,
        'numTemplateFrames': template_frames,
        'numBins': V_static.shape[0],
        'numFrames': V_static.shape[1],
    }
    NMFparam = {
        'numComp': 2,
        'numIter': 80,
        'numBins': V_static.shape[0],
        'numFrames': V_static.shape[1],
        'initW': np.random.rand(V_static.shape[0], 2),
        'initH': np.random.rand(2, V_static.shape[1]),
        'costFunc': 'KLDiv',
        'fixW': False,
    }

    static_W_nmf, static_H_nmf, nmfV = NMF(V_static.copy(), parameter=NMFparam)
    static_W_nmfd, static_H_nmfd, nmfdV, cost, tensorW = NMFD(V_static.copy(), parameter=NMFDparam)
    
    static_mix_phase = np.angle(static_mix_stft)

    for i, comp_mag in enumerate(nmfdV):
        comp_mag_rescaled = comp_mag * V_static_scale
        comp_spec = comp_mag_rescaled * np.exp(1j * static_mix_phase)
        y_i = librosa.istft(comp_spec, hop_length=HOP, length=len(static_mix))
        if i == 0:
            sf.write(f"NMFDresults/static_kick/{static_mix_file.stem}_kick.wav", y_i, sample_rate)
        else: 
            sf.write(f"NMFDresults/static_snare/{static_mix_file.stem}_snare.wav", y_i, sample_rate)

    for i, comp_mag in enumerate(nmfV):
        comp_mag_rescaled = comp_mag * V_static_scale
        comp_spec = comp_mag_rescaled * np.exp(1j * static_mix_phase)
        y_i = librosa.istft(comp_spec, hop_length=HOP, length=len(static_mix))
                
        if i == 0:
            sf.write(f"NMFresults/static_kick/{static_mix_file.stem}_kick.wav", y_i, sample_rate)
        else: 
            sf.write(f"NMFresults/static_snare/{static_mix_file.stem}_snare.wav", y_i, sample_rate)

dataset/static/static_mix_000.wav


Processing:   0%|          | 0/80 [00:00<?, ?it/s]

Processing:   0%|          | 0/80 [00:00<?, ?it/s]

In [35]:
#run NMF and NMFD on dynamic mixtures
for dynamic_mix_file in sorted(Path("dataset/dynamic").glob("*.wav"), key=lambda x: int(x.stem.split('_')[-1])):
    print(dynamic_mix_file)
    dynamic_mix, sr = librosa.load(dynamic_mix_file, sr=sample_rate)
    dynamic_mix_stft = librosa.stft(dynamic_mix, n_fft=FRAME, hop_length=HOP)
    V_dynamic = np.abs(dynamic_mix_stft) + EPS
    V_dynamic_scale = V_dynamic.sum()

    # Build parameters matching this file's spectrogram shape
    NMFDparam = {
        'numComp': 2,
        'numIter': 80,
        'numTemplateFrames': template_frames,
        'numBins': V_dynamic.shape[0],
        'numFrames': V_dynamic.shape[1],
    }
    NMFparam = {
        'numComp': 2,
        'numIter': 80,
        'numBins': V_dynamic.shape[0],
        'numFrames': V_dynamic.shape[1],
        'initW': np.random.rand(V_dynamic.shape[0], 2),
        'initH': np.random.rand(2, V_dynamic.shape[1]),
        'costFunc': 'KLDiv',
        'fixW': False,
    }

    dynamic_W_nmf, dynamic_H_nmf, nmfV = NMF(V_dynamic.copy(), parameter=NMFparam)
    dynamic_W_nmfd, dynamic_H_nmfd, nmfdV, cost, tensorW = NMFD(V_dynamic.copy(), parameter=NMFDparam)
    
    dynamic_mix_phase = np.angle(dynamic_mix_stft)

    for i, comp_mag in enumerate(nmfdV):
        comp_mag_rescaled = comp_mag * V_dynamic_scale
        comp_spec = comp_mag_rescaled * np.exp(1j * dynamic_mix_phase)
        y_i = librosa.istft(comp_spec, hop_length=HOP, length=len(dynamic_mix))

        if i == 0:
            sf.write(f"NMFDresults/dynamic_kick/{dynamic_mix_file.stem}_kick.wav", y_i, sample_rate)
        else: 
            sf.write(f"NMFDresults/dynamic_snare/{dynamic_mix_file.stem}_snare.wav", y_i, sample_rate)

    for i, comp_mag in enumerate(nmfV):
        comp_mag_rescaled = comp_mag * V_dynamic_scale
        comp_spec = comp_mag_rescaled * np.exp(1j * dynamic_mix_phase)
        y_i = librosa.istft(comp_spec, hop_length=HOP, length=len(dynamic_mix))
        
        if i == 0:
            sf.write(f"NMFresults/dynamic_kick/{dynamic_mix_file.stem}_kick.wav", y_i, sample_rate)
        else: 
            sf.write(f"NMFresults/dynamic_snare/{dynamic_mix_file.stem}_snare.wav", y_i, sample_rate)

dataset/dynamic/dynamic_mix_000.wav


Processing:   0%|          | 0/80 [00:00<?, ?it/s]

Processing:   0%|          | 0/80 [00:00<?, ?it/s]

### Metrics

In [32]:
#calculate metrics
all_static_nmf_sdr = []
all_static_nmf_sir = []
all_static_nmf_sar = []
all_static_nmfd_sdr = []
all_static_nmfd_sir = []
all_static_nmfd_sar = []
all_dynamic_nmf_sdr = []
all_dynamic_nmf_sir = []
all_dynamic_nmf_sar = []
all_dynamic_nmfd_sdr = []
all_dynamic_nmfd_sir = []
all_dynamic_nmfd_sar = []

for i in range(num_files):
    # Load reference sources (ground truth)
    static_reference_snare, _ = librosa.load(f'dataset/static_snare/static_snare_{i:03d}.wav', sr=sample_rate)
    static_reference_kick, _ = librosa.load(f'dataset/static_kick/static_kick_{i:03d}.wav', sr=sample_rate)
    static_reference_sources = np.array([static_reference_kick, static_reference_snare])

    dynamic_reference_snare, _ = librosa.load(f'dataset/dynamic_snare/dynamic_snare_{i:03d}.wav', sr=sample_rate)
    dynamic_reference_kick, _ = librosa.load(f'dataset/dynamic_kick/dynamic_kick_{i:03d}.wav', sr=sample_rate)
    dynamic_reference_sources = np.array([dynamic_reference_kick, dynamic_reference_snare])

    # Load static NMF estimated sources
    static_nmf_kick, _ = librosa.load(f'NMFresults/static_kick/static_mix_{i:03d}_kick.wav', sr=sample_rate)
    static_nmf_snare, _ = librosa.load(f'NMFresults/static_snare/static_mix_{i:03d}_snare.wav', sr=sample_rate)
    static_nmf_estimated_sources = np.array([static_nmf_kick, static_nmf_snare])

    # Load static NMFD estimated sources
    static_nmfd_kick, _ = librosa.load(f'NMFDresults/static_kick/static_mix_{i:03d}_kick.wav', sr=sample_rate)
    static_nmfd_snare, _ = librosa.load(f'NMFDresults/static_snare/static_mix_{i:03d}_snare.wav', sr=sample_rate)
    static_nmfd_estimated_sources = np.array([static_nmfd_kick, static_nmfd_snare])

    # Load dynamic NMF estimated sources
    dynamic_nmf_kick, _ = librosa.load(f'NMFresults/dynamic_kick/dynamic_mix_{i:03d}_kick.wav', sr=sample_rate)
    dynamic_nmf_snare, _ = librosa.load(f'NMFresults/dynamic_snare/dynamic_mix_{i:03d}_snare.wav', sr=sample_rate)
    dynamic_nmf_estimated_sources = np.array([dynamic_nmf_kick, dynamic_nmf_snare])

    # Load dynamic NMFD estimated sources
    dynamic_nmfd_kick, _ = librosa.load(f'NMFDresults/dynamic_kick/dynamic_mix_{i:03d}_kick.wav', sr=sample_rate)
    dynamic_nmfd_snare, _ = librosa.load(f'NMFDresults/dynamic_snare/dynamic_mix_{i:03d}_snare.wav', sr=sample_rate)
    dynamic_nmfd_estimated_sources = np.array([dynamic_nmfd_kick, dynamic_nmfd_snare])

    # Ensure static arrays have the same length
    static_min_len = min(static_reference_sources.shape[1], static_nmf_estimated_sources.shape[1], static_nmfd_estimated_sources.shape[1])
    static_reference_sources = static_reference_sources[:, :static_min_len]
    static_nmf_estimated_sources = static_nmf_estimated_sources[:, :static_min_len]
    static_nmfd_estimated_sources = static_nmfd_estimated_sources[:, :static_min_len]

    # Ensure dynamic arrays have the same length
    dynamic_min_len = min(dynamic_reference_sources.shape[1], dynamic_nmf_estimated_sources.shape[1], dynamic_nmfd_estimated_sources.shape[1])
    dynamic_reference_sources = dynamic_reference_sources[:, :dynamic_min_len]
    dynamic_nmf_estimated_sources = dynamic_nmf_estimated_sources[:, :dynamic_min_len]
    dynamic_nmfd_estimated_sources = dynamic_nmfd_estimated_sources[:, :dynamic_min_len]

    # Evaluate static against static references
    # bss_eval_sources returns metrics already ordered by reference index [kick, snare]
    static_nmf_sdr, static_nmf_sir, static_nmf_sar, _ = bss_eval_sources(static_reference_sources, static_nmf_estimated_sources)
    static_nmfd_sdr, static_nmfd_sir, static_nmfd_sar, _ = bss_eval_sources(static_reference_sources, static_nmfd_estimated_sources)

    # Evaluate dynamic against dynamic references
    dynamic_nmf_sdr, dynamic_nmf_sir, dynamic_nmf_sar, _ = bss_eval_sources(dynamic_reference_sources, dynamic_nmf_estimated_sources)
    dynamic_nmfd_sdr, dynamic_nmfd_sir, dynamic_nmfd_sar, _ = bss_eval_sources(dynamic_reference_sources, dynamic_nmfd_estimated_sources)

    all_static_nmf_sdr.append(static_nmf_sdr)
    all_static_nmf_sir.append(static_nmf_sir)
    all_static_nmf_sar.append(static_nmf_sar)
    all_static_nmfd_sdr.append(static_nmfd_sdr)
    all_static_nmfd_sir.append(static_nmfd_sir)
    all_static_nmfd_sar.append(static_nmfd_sar)
    all_dynamic_nmf_sdr.append(dynamic_nmf_sdr)
    all_dynamic_nmf_sir.append(dynamic_nmf_sir)
    all_dynamic_nmf_sar.append(dynamic_nmf_sar)
    all_dynamic_nmfd_sdr.append(dynamic_nmfd_sdr)
    all_dynamic_nmfd_sir.append(dynamic_nmfd_sir)
    all_dynamic_nmfd_sar.append(dynamic_nmfd_sar)

    print(f"\n{i:03d} - Static NMF SDR: {static_nmf_sdr}")
    print(f"{i:03d} - Static NMF SIR: {static_nmf_sir}")
    print(f"{i:03d} - Static NMF SAR: {static_nmf_sar}")
    print(f"\n{i:03d} - Static NMFD SDR: {static_nmfd_sdr}")
    print(f"{i:03d} - Static NMFD SIR: {static_nmfd_sir}")
    print(f"{i:03d} - Static NMFD SAR: {static_nmfd_sar}")
    print(f"\n{i:03d} - Dynamic NMF SDR: {dynamic_nmf_sdr}")
    print(f"{i:03d} - Dynamic NMF SIR: {dynamic_nmf_sir}")
    print(f"{i:03d} - Dynamic NMF SAR: {dynamic_nmf_sar}")
    print(f"\n{i:03d} - Dynamic NMFD SDR: {dynamic_nmfd_sdr}")
    print(f"{i:03d} - Dynamic NMFD SIR: {dynamic_nmfd_sir}")
    print(f"{i:03d} - Dynamic NMFD SAR: {dynamic_nmfd_sar}")

/var/folders/zs/xdh0v3x55sz09h0zynkg0lz80000gq/T/ipykernel_6409/276895809.py:26: UserWarning: PySoundFile failed. Trying audioread instead.
  static_nmf_kick, _ = librosa.load(f'NMFresults/static_kick/static_mix_{i:03d}_kick.wav', sr=sample_rate)
/Users/micahlim/Desktop/NMF_separation/.venv/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: 'NMFresults/static_kick/static_mix_000_kick.wav'

In [33]:
#calculate metrics averages
static_nmf_sdr_average = sum(all_static_nmf_sdr) / len(all_static_nmf_sdr)
static_nmf_sir_average = sum(all_static_nmf_sir) / len(all_static_nmf_sir)
static_nmf_sar_average = sum(all_static_nmf_sar) / len(all_static_nmf_sar)

static_nmfd_sdr_average = sum(all_static_nmfd_sdr) / len(all_static_nmfd_sdr)
static_nmfd_sir_average = sum(all_static_nmfd_sir) / len(all_static_nmfd_sir)
static_nmfd_sar_average = sum(all_static_nmfd_sar) / len(all_static_nmfd_sar)

dynamic_nmf_sdr_average = sum(all_dynamic_nmf_sdr) / len(all_dynamic_nmf_sdr)
dynamic_nmf_sir_average = sum(all_dynamic_nmf_sir) / len(all_dynamic_nmf_sir)
dynamic_nmf_sar_average = sum(all_dynamic_nmf_sar) / len(all_dynamic_nmf_sar)

dynamic_nmfd_sdr_average = sum(all_dynamic_nmfd_sdr) / len(all_dynamic_nmfd_sdr)
dynamic_nmfd_sir_average = sum(all_dynamic_nmfd_sir) / len(all_dynamic_nmfd_sir)
dynamic_nmfd_sar_average = sum(all_dynamic_nmfd_sar) / len(all_dynamic_nmfd_sar)

print("Averages: \n")
print("Static NMF Average SDR:", static_nmf_sdr_average)
#print("Static NMF Average SIR:", static_nmf_sir_average)
#print("Static NMF Average SAR:", static_nmf_sar_average)
print("Static NMFD Average SDR:", static_nmfd_sdr_average)
#print("Static NMFD Average SIR:", static_nmfd_sir_average)
#print("Static NMFD Average SAR:", static_nmfd_sar_average)
print("Dynamic NMF Average SDR:", dynamic_nmf_sdr_average)
#print("Dynamic NMF Average SIR:", dynamic_nmf_sir_average)
#print("Dynamic NMF Average SAR:", dynamic_nmf_sar_average)
print("Dynamic NMFD Average SDR:", dynamic_nmfd_sdr_average)
#print("Dynamic NMFD Average SIR:", dynamic_nmfd_sir_average)
#print("Dynamic NMFD Average SAR:", dynamic_nmfd_sar_average)

ZeroDivisionError: division by zero

### Plot

In [37]:

def plot_stem_spectrograms(num_files, sample_rate, FRAME, HOP):
    """Plot spectrograms and magnitude/activations for each dataset stem."""
    rms_global_max = -np.inf  # will hold the global max (in dB) across all RMS envelopes and all files

    for i in range(num_files):
        static_kick, _ = librosa.load(f'dataset/static_kick/static_kick_{i:03d}.wav', sr=sample_rate)
        static_snare, _ = librosa.load(f'dataset/static_snare/static_snare_{i:03d}.wav', sr=sample_rate)
        dynamic_kick, _ = librosa.load(f'dataset/dynamic_kick/dynamic_kick_{i:03d}.wav', sr=sample_rate)
        dynamic_snare, _ = librosa.load(f'dataset/dynamic_snare/dynamic_snare_{i:03d}.wav', sr=sample_rate)

        stems = [
            ('Static Kick', static_kick),
            ('Static Snare', static_snare),
            ('Dynamic Kick', dynamic_kick),
            ('Dynamic Snare', dynamic_snare),
        ]

        # --- First pass: compute global RMS max (in dB) across all stems ---
        all_rms_db = []
        for label, audio in stems:
            stft = librosa.stft(audio, n_fft=FRAME, hop_length=HOP)
            mag = np.abs(stft)
            rms = np.sqrt(np.mean(mag ** 2, axis=0))
            rms_db = 20 * np.log10(rms + 1e-6)
            all_rms_db.append(rms_db)
        file_rms_db_max = max(r.max() for r in all_rms_db)
        rms_global_max = max(rms_global_max, file_rms_db_max)

    # --- Second pass: plot with uniform y-axis ---
    for i in range(num_files):
        static_kick, _ = librosa.load(f'dataset/static_kick/static_kick_{i:03d}.wav', sr=sample_rate)
        static_snare, _ = librosa.load(f'dataset/static_snare/static_snare_{i:03d}.wav', sr=sample_rate)
        dynamic_kick, _ = librosa.load(f'dataset/dynamic_kick/dynamic_kick_{i:03d}.wav', sr=sample_rate)
        dynamic_snare, _ = librosa.load(f'dataset/dynamic_snare/dynamic_snare_{i:03d}.wav', sr=sample_rate)

        stems = [
            ('Static Kick', static_kick),
            ('Static Snare', static_snare),
            ('Dynamic Kick', dynamic_kick),
            ('Dynamic Snare', dynamic_snare),
        ]

        fig, axes = plt.subplots(4, 3, figsize=(18, 16))
        fig.suptitle(f'Stem Analysis — File {i:03d}', fontsize=16, fontweight='bold')

        freqs = librosa.fft_frequencies(sr=sample_rate, n_fft=FRAME)

        for row, (label, audio) in enumerate(stems):
            # Compute STFT
            stft = librosa.stft(audio, n_fft=FRAME, hop_length=HOP)
            mag = np.abs(stft)
            mag_db = librosa.amplitude_to_db(mag, ref=np.max)

            # --- Spectrogram ---
            img = librosa.display.specshow(
                mag_db, sr=sample_rate, hop_length=HOP, n_fft=FRAME,
                x_axis='time', y_axis='hz', ax=axes[row, 0], cmap='magma'
            )
            axes[row, 0].set_title(f'{label} — Spectrogram')
            axes[row, 0].set_ylabel('Frequency (Hz)')
            axes[row, 0].set_xlabel('Time (s)')
            fig.colorbar(img, ax=axes[row, 0], format='%+2.0f dB')

            # --- Average magnitude spectrum (W-like) ---
            avg_mag = mag.mean(axis=1)
            axes[row, 1].plot(freqs, avg_mag, color='tab:blue')
            axes[row, 1].set_title(f'{label} — Avg Magnitude Spectrum')
            axes[row, 1].set_xlabel('Frequency (Hz)')
            axes[row, 1].set_ylabel('Magnitude')
            axes[row, 1].set_xlim([0, sample_rate / 2])

            # --- Activation envelope (RMS over time, H-like) in dB ---
            times = librosa.frames_to_time(np.arange(mag.shape[1]), sr=sample_rate, hop_length=HOP)
            rms = np.sqrt(np.mean(mag ** 2, axis=0))
            rms_db = 20 * np.log10(rms + 1e-6)
            axes[row, 2].plot(times, rms_db, color='tab:orange')
            axes[row, 2].set_title(f'{label} — Activation (RMS Envelope)')
            axes[row, 2].set_xlabel('Time (s)')
            axes[row, 2].set_ylabel('RMS Magnitude (dB)')
            axes[row, 2].set_ylim(None, rms_global_max + 3)

        plt.tight_layout()
        plt.savefig(f'original_data_plot/stem_spectrogram_analysis_{i:03d}.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f'Saved stem_spectrogram_analysis_{i:03d}.png')

    print(f'Global RMS max = {rms_global_max:.4f}')
    return rms_global_max


# Call the function
rms_global_max = plot_stem_spectrograms(num_files, sample_rate, FRAME, HOP)


Saved stem_spectrogram_analysis_000.png
Global RMS max = 20.1450


In [30]:
#plotting functions

def plot_nmf_decomposition(W, H, title="NMF Decomposition", sr=22050, hop_length=256, n_fft=512, save_path=None, h_ylim=None):
    """Plot W (templates) and H (activations) from NMF.
    
    Parameters
    ----------
    W : np.ndarray, shape (numBins, numComp)
    H : np.ndarray, shape (numComp, numFrames)
    title : str
    sr : int
    hop_length : int
    n_fft : int
    save_path : str or None
        If provided, save the figure to this path instead of showing it.
    h_ylim : float or None
        If provided, set the y-axis upper limit for all H activation subplots (in dB).
    """
    numComp = W.shape[1]
    fig, axes = plt.subplots(numComp, 2, figsize=(14, 3 * numComp))
    if numComp == 1:
        axes = axes[np.newaxis, :]
    
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    times = librosa.frames_to_time(np.arange(H.shape[1]), sr=sr, hop_length=hop_length)
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    for r in range(numComp):
        # Plot W column (spectral template)
        axes[r, 0].plot(freqs, 20 * np.log10(W[:, r] + 1e-6))  # Convert to dB
        axes[r, 0].set_title(f'W — Component {r}')
        axes[r, 0].set_xlabel('Frequency (Hz)')
        axes[r, 0].set_ylabel('Magnitude (dB)')
        
        # Plot H row (activation over time) in dB
        axes[r, 1].plot(times, 20 * np.log10(H[r, :] + 1e-6))
        axes[r, 1].set_title(f'H — Component {r}')
        axes[r, 1].set_xlabel('Time (s)')
        axes[r, 1].set_ylabel('Activation (dB)')
        if h_ylim is not None:
            axes[r, 1].set_ylim(h_ylim)
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()


def plot_nmfd_decomposition(W_list, H, title="NMFD Decomposition", sr=22050, hop_length=256, n_fft=512, save_path=None, h_ylim=None):
    """Plot all Wt templates and H activations from NMFD.
    
    Parameters
    ----------
    W_list : list of np.ndarray, each shape (numBins, numTemplateFrames)
    H : np.ndarray, shape (numComp, numFrames)
    title : str
    sr : int
    hop_length : int
    n_fft : int
    save_path : str or None
        If provided, save the figure to this path instead of showing it.
    h_ylim : float or None
        If provided, set the y-axis upper limit for all H activation subplots (in dB).
    """
    numComp = len(W_list)
    numTemplateFrames = W_list[0].shape[1]

    fig, axes = plt.subplots(numComp, 2, figsize=(14, 4 * numComp))
    if numComp == 1:
        axes = axes[np.newaxis, :]

    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    times = librosa.frames_to_time(np.arange(H.shape[1]), sr=sr, hop_length=hop_length)
    template_times = librosa.frames_to_time(np.arange(numTemplateFrames), sr=sr, hop_length=hop_length)

    fig.suptitle(title, fontsize=14, fontweight='bold')

    for r in range(numComp):
        # Plot 2D template as a heatmap (freq x template time frames)
        im = axes[r, 0].pcolormesh(template_times, freqs, 20 * np.log10(W_list[r] + 1e-6), shading='auto', cmap='magma')  # Convert to dB
        axes[r, 0].set_title(f"W — Component {r} (2D template)")
        axes[r, 0].set_xlabel("Template frame time (s)")
        axes[r, 0].set_ylabel("Frequency (Hz)")
        fig.colorbar(im, ax=axes[r, 0])

        # Plot H row (activation over time) in dB
        axes[r, 1].plot(times, 20 * np.log10(H[r, :] + 1e-6))
        axes[r, 1].set_title(f"H — Component {r}")
        axes[r, 1].set_xlabel("Time (s)")
        axes[r, 1].set_ylabel("Activation (dB)")
        if h_ylim is not None:
            axes[r, 1].set_ylim(h_ylim)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()

# Ensure the heatmap in plot_all_decompositions also reflects templateFrames

def plot_all_decompositions(s_W_nmf, s_H_nmf, s_W_nmfd, s_H_nmfd,
                            d_W_nmf, d_H_nmf, d_W_nmfd, d_H_nmfd,
                            file_idx, sr=22050, hop_length=256, n_fft=512,
                            save_path=None, h_ylim=None):
    """Plot Static NMF, Static NMFD, Dynamic NMF, Dynamic NMFD in one figure.
    
    Layout: 4 sections (rows grouped by method), each with W and H subplots.
    
    Parameters
    ----------
    h_ylim : float or None
        If provided, set the y-axis limits (min, max) for all H activation subplots (in dB).
    """
    numComp = s_W_nmf.shape[1]
    # 4 methods x numComp rows, 2 columns (W, H)
    nrows = 4 * numComp
    fig, axes = plt.subplots(nrows, 2, figsize=(16, 3 * nrows))
    fig.suptitle(f'File {file_idx:03d} — All Decompositions', fontsize=16, fontweight='bold', y=1.0)

    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

    # Helper to plot NMF W/H into a block of rows
    def _plot_nmf(W, H, row_start, label):
        times = librosa.frames_to_time(np.arange(H.shape[1]), sr=sr, hop_length=hop_length)
        for r in range(numComp):
            row = row_start + r
            axes[row, 0].plot(freqs, 20 * np.log10(W[:, r] + 1e-6))  # Convert to dB
            axes[row, 0].set_title(f'{label} — W Component {r}')
            axes[row, 0].set_xlabel('Frequency (Hz)')
            axes[row, 0].set_ylabel('Magnitude (dB)')
            axes[row, 1].plot(times, 20 * np.log10(H[r, :] + 1e-6))  # Convert to dB
            axes[row, 1].set_title(f'{label} — H Component {r}')
            axes[row, 1].set_xlabel('Time (s)')
            axes[row, 1].set_ylabel('Activation (dB)')
            if h_ylim is not None:
                axes[row, 1].set_ylim(h_ylim)

    # Helper to plot NMFD W_list/H into a block of rows
    def _plot_nmfd(W_list, H, row_start, label):
        times = librosa.frames_to_time(np.arange(H.shape[1]), sr=sr, hop_length=hop_length)
        numTF = W_list[0].shape[1]
        template_times = librosa.frames_to_time(np.arange(numTF), sr=sr, hop_length=hop_length)
        for r in range(numComp):
            row = row_start + r
            im = axes[row, 0].pcolormesh(template_times, freqs, 20 * np.log10(W_list[r] + 1e-6), shading='auto', cmap='magma')  # Convert to dB
            axes[row, 0].set_title(f'{label} — W Component {r} (2D template)')
            axes[row, 0].set_xlabel('Template frame time (s)')
            axes[row, 0].set_ylabel('Frequency (Hz)')
            fig.colorbar(im, ax=axes[row, 0])
            axes[row, 1].plot(times, 20 * np.log10(H[r, :] + 1e-6))  # Convert to dB
            axes[row, 1].set_title(f'{label} — H Component {r}')
            axes[row, 1].set_xlabel('Time (s)')
            axes[row, 1].set_ylabel('Activation (dB)')
            if h_ylim is not None:
                axes[row, 1].set_ylim(h_ylim)

    _plot_nmf(s_W_nmf, s_H_nmf, 0, 'Static NMF')
    _plot_nmfd(s_W_nmfd, s_H_nmfd, numComp, 'Static NMFD')
    _plot_nmf(d_W_nmf, d_H_nmf, 2 * numComp, 'Dynamic NMF')
    _plot_nmfd(d_W_nmfd, d_H_nmfd, 3 * numComp, 'Dynamic NMFD')

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()

In [31]:
# Save plots as pngs
os.makedirs('results_plot', exist_ok=True)

for i in range(num_files):
    print(f"Processing file {i:03d}...")
    
    # --- Static ---
    static_mix, _ = librosa.load(f'dataset/static/static_mix_{i:03d}.wav', sr=sample_rate)
    static_mix_stft = librosa.stft(static_mix, n_fft=FRAME, hop_length=HOP)
    V_static = np.abs(static_mix_stft) + EPS

    NMFDparam_s = {
        'numComp': 2, 'numIter': 80, 'numTemplateFrames': template_frames,
        'numBins': V_static.shape[0], 'numFrames': V_static.shape[1],
    }
    NMFparam_s = {
        'numComp': 2, 'numIter': 80,
        'numBins': V_static.shape[0], 'numFrames': V_static.shape[1],
        'initW': np.random.rand(V_static.shape[0], 2),
        'initH': np.random.rand(2, V_static.shape[1]),
        'costFunc': 'KLDiv', 'fixW': False,
    }

    s_W_nmf, s_H_nmf, _ = NMF(V_static.copy(), parameter=NMFparam_s)
    s_W_nmfd, s_H_nmfd, _, _, _ = NMFD(V_static.copy(), parameter=NMFDparam_s)

    # --- Dynamic ---
    dynamic_mix, _ = librosa.load(f'dataset/dynamic/dynamic_mix_{i:03d}.wav', sr=sample_rate)
    dynamic_mix_stft = librosa.stft(dynamic_mix, n_fft=FRAME, hop_length=HOP)
    V_dynamic = np.abs(dynamic_mix_stft) + EPS

    NMFDparam_d = {
        'numComp': 2, 'numIter': 80, 'numTemplateFrames': 8,
        'numBins': V_dynamic.shape[0], 'numFrames': V_dynamic.shape[1],
    }
    NMFparam_d = {
        'numComp': 2, 'numIter': 80,
        'numBins': V_dynamic.shape[0], 'numFrames': V_dynamic.shape[1],
        'initW': np.random.rand(V_dynamic.shape[0], 2),
        'initH': np.random.rand(2, V_dynamic.shape[1]),
        'costFunc': 'KLDiv', 'fixW': False,
    }

    d_W_nmf, d_H_nmf, _ = NMF(V_dynamic.copy(), parameter=NMFparam_d)
    d_W_nmfd, d_H_nmfd, _, _, _ = NMFD(V_dynamic.copy(), parameter=NMFDparam_d)

    # Convert rms_global_max to dB for uniform y-axis on H activations
    h_ylim_db = (20 * np.log10(1e-6), 20 * np.log10(rms_global_max * 1.05 + 1e-6))

    plot_all_decompositions(
        s_W_nmf, s_H_nmf, s_W_nmfd, s_H_nmfd,
        d_W_nmf, d_H_nmf, d_W_nmfd, d_H_nmfd,
        file_idx=i, sr=sample_rate, hop_length=HOP, n_fft=FRAME,
        save_path=f"results_plot/decomposition_{i:03d}.png",
        h_ylim=h_ylim_db,
    )

print("Done — combined plots saved to results_plot/")

Processing file 000...


Processing:   0%|          | 0/80 [00:00<?, ?it/s]

Processing:   0%|          | 0/80 [00:00<?, ?it/s]

Processing:   0%|          | 0/80 [00:00<?, ?it/s]

Processing:   0%|          | 0/80 [00:00<?, ?it/s]

Done — combined plots saved to results_plot/
